In [0]:
%run ../common/config


In [0]:
print(env_catalog)

In [0]:
env_schema="bronze"
#print(schema)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {env_schema}")
spark.sql(f"USE SCHEMA {env_schema}")


In [0]:
files = [
    f"https://raw.githubusercontent.com/v889/Healthcare-Claims-Intelligence-Platform/main/data/claimsTrans/claims_transactions_{i}.csv"
    for i in range(1, 11)
]


In [0]:
files=[files[0]]+files[2:]

In [0]:
print(files)

In [0]:
import pandas as pd

dfs = []

for file_url in files:
    pdf = pd.read_csv(file_url)

    pdf["source_file"] = file_url.split("/")[-1]

    dfs.append(pdf)

combined_pdf = pd.concat(dfs, ignore_index=True)

In [0]:
claims_df = spark.createDataFrame(combined_pdf)

display(claims_df)

In [0]:
claims_df.printSchema()

In [0]:
claims_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        f"{catalog}.bronze.claims_transactions"
    )

In [0]:
source_count = claims_df.count()

target_count = spark.sql(
    f"""
    SELECT COUNT(*)
    FROM {catalog}.bronze.claims_transactions
    """
).collect()[0][0]

print(f"Source Count : {source_count}")
print(f"Target Count : {target_count}")